# Mitigation Impact Report

Compare latency distributions before and after mitigations (E7 → E13 → E14 → E15).
Generate the mitigation waterfall analysis from Blueprint §08.

**Blueprint Reference:** §08 Mitigations, Fig 7 (Mitigation Waterfall)

In [ ]:
import json, csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DATA_DIR = Path('../data')

def load_latencies(exp_name, run_idx=0):
    exp_dir = DATA_DIR / exp_name
    jsons = sorted(exp_dir.glob('*.json'))
    if run_idx >= len(jsons): return None
    with open(jsons[run_idx]) as f:
        d = json.load(f)
    details = d.get('details', [])
    return np.array([det['latency']/1e6 for det in details if det.get('status')=='OK' and det.get('latency',0)>0])

## 1. Mitigation Waterfall

In [ ]:
MITIGATION_CHAIN = [
    ('e7-full-stress', 'Baseline (stressed)'),
    ('e13-cpu-pinning', '+CPU Pinning (M1)'),
    ('e15-full-isolation', '+Full Isolation (M1+M2+M3+M4)'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: p99 reduction
names = []
p99s = []
for exp, label in MITIGATION_CHAIN:
    lats = load_latencies(exp)
    if lats is not None:
        names.append(label)
        p99s.append(np.percentile(lats, 99))

baseline_p99 = p99s[0] if p99s else 1
reductions = [(1 - p/baseline_p99) * 100 for p in p99s]

colors = ['#e74c3c'] + ['#2ecc71'] * (len(p99s) - 1)
ax1.barh(range(len(names)), p99s, color=colors, edgecolor='white')
for i, (p, r) in enumerate(zip(p99s, reductions)):
    ax1.text(p + 5, i, f'{p:.0f}ms ({r:+.0f}%)', va='center', fontsize=9)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, fontsize=9)
ax1.set_xlabel('p99 Latency (ms)')
ax1.set_title('Mitigation Waterfall — p99 Reduction')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# CDF overlay
for exp, label in MITIGATION_CHAIN:
    lats = load_latencies(exp)
    if lats is not None:
        sorted_l = np.sort(lats)
        cdf = np.arange(1, len(sorted_l)+1) / len(sorted_l)
        ax2.plot(sorted_l, cdf, label=f'{label} (n={len(lats)})', linewidth=2)
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('CDF')
ax2.set_title('CDF: Before and After Mitigations')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig.suptitle('Mitigation Impact Analysis — Blueprint §08', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Signal Attribution: Before vs After

In [ ]:
# Load per-signal Mann-Whitney results
per_signal = []
ps_path = Path('../stats/per_signal_mannwhitney.csv')
if ps_path.exists():
    with open(ps_path) as f:
        per_signal = list(csv.DictReader(f))

# Compare wakeup_delay across mitigation chain
mit_exps = ['e7-full-stress', 'e13-cpu-pinning', 'e15-full-isolation']
wakeup_data = [r for r in per_signal if r['signal'] == 'wakeup_delay_p99' and r['experiment'] in mit_exps]

if wakeup_data:
    fig, ax = plt.subplots(figsize=(10, 4))
    exp_names = [r['experiment'] for r in wakeup_data]
    deltas = [float(r['cliffs_delta']) for r in wakeup_data]
    ratios = [float(r['case_control_ratio']) for r in wakeup_data]
    
    x = np.arange(len(exp_names))
    ax.bar(x - 0.2, deltas, 0.35, label="Cliff's δ", color='steelblue')
    ax.bar(x + 0.2, ratios, 0.35, label='Case/Control Ratio', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(exp_names, fontsize=9)
    ax.legend()
    ax.set_ylabel('Value')
    ax.set_title('Wakeup Delay Signal: Stress → Pinning → Isolation')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No per-signal data — run per_signal_mannwhitney.py first')

## 3. Chi-Squared Throttle Results

In [ ]:
chi2_path = Path('../stats/chi_squared_throttle.csv')
if chi2_path.exists():
    with open(chi2_path) as f:
        chi2_data = list(csv.DictReader(f))
    
    print(f'{"Experiment":<30} {"χ²":>8} {"p":>8} {"V":>8} {"Sig?":>6}')
    print('─' * 65)
    for r in chi2_data:
        sig = '★' if r.get('significant', 'False') == 'True' else ''
        print(f"{r['experiment']:<30} {float(r['chi2']):>8.2f} {float(r['p_value']):>8.4f} "
              f"{float(r['cramers_v']):>8.3f} {sig:>6}")
else:
    print('No chi-squared data — run chi_squared_throttle.py first')